# 🔴 Solution: Implement LayerNorm（面试版）

## 📋 题目描述

**难度：** Medium

实现层归一化。

LayerNorm 对每个样本沿特征维度进行归一化，不依赖批大小即可稳定训练。

**签名:** `my_layer_norm(x, gamma, beta, eps=1e-5) -> Tensor`

**参数:**
- `x` — 输入张量 (..., D)
- `gamma` — 缩放参数 (D,)
- `beta` — 偏移参数 (D,)
- `eps` — 数值稳定性的 epsilon

**返回:** 归一化后的张量，形状与 x 相同

**约束:**
- 沿最后一个维度归一化
- 方差使用 `unbiased=False`
- 必须与 `F.layer_norm` 一致

**提示：** 1. `mean = x.mean(dim=-1, keepdim=True)`
2. `var = x.var(dim=-1, keepdim=True, unbiased=False)`
3. `x_norm = (x - mean) / sqrt(var + eps)` → `gamma * x_norm + beta`


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def layernorm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    # ---- Step 1: 沿最后一维求均值 ----
    # keepdim=True：[32, 768] → [32, 1]（而非 [32]），方便广播
    mean = x.mean(dim=-1, keepdim=True)

    # ---- Step 2: 沿最后一维求方差 ----
    # 用 (x - mean)² 的均值，这是总体方差（除以 N，非 N-1）
    # 面试关键：为什么不用无偏估计（除以 N-1）？
    # A: 深度学习中 N 通常很大，N vs N-1 差异可忽略，且 PyTorch 默认用有偏
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

    # ---- Step 3: 归一化 ----
    # 标准化公式：(x - μ) / √(σ² + ε)
    # eps 的作用：防止方差为0时除零（例如全常数输入）
    x_norm = (x - mean) / torch.sqrt(var + eps)

    # ---- Step 4: 仿射变换 ----
    # γ * x_norm + β：可学习的缩放和偏移
    # 为什么要这一步？纯归一化会限制网络表达能力
    # γ, β 让网络可以学习"需要多少归一化"
    return weight * x_norm + bias

# 面试追问：
# Q: LayerNorm vs BatchNorm 区别？
# A: LN 沿特征维度归一化（每个样本独立），BN 沿 batch 维度归一化
# Q: 为什么 Transformer 用 LN 不用 BN？
# A: 序列长度可变，batch 统计不稳定；LN 不依赖 batch，更稳定

In [ ]:
# Verify
x = torch.randn(2, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)
out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)
print("Match ref?", torch.allclose(out, ref, atol=1e-4))

In [ ]:
# Run judge
from torch_judge import check
check("layernorm")

## 📝 核心思路总结

1. **归一化公式**：(x - μ) / √(σ² + ε)，沿最后维度计算统计量
2. **keepdim 的必要性**：保持维度用于广播，避免形状错误
3. **仿射变换**：γ, β 可学习参数，让网络恢复表达能力
4. **eps 的作用**：数值稳定，防止除零